# Arle LSTM Training
**Arlecchino IDE — Code Completion Model**

This notebook trains a lightweight LSTM model for code completion.

## Steps:
1. Install dependencies
2. Download CodeSearchNet dataset
3. Build BPE tokenizer
4. Train LSTM model
5. Export to ONNX
6. Download the model

**Estimated time: 8-12 hours on free T4 GPU**

## Step 0: Enable GPU

**IMPORTANT:** Go to `Runtime` → `Change runtime type` → Select `T4 GPU` → Save

Then run the cell below to verify:

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU! Go to Runtime -> Change runtime type -> T4 GPU")

## Step 1: Install Dependencies

In [ ]:
!pip install -q datasets tokenizers onnx onnxruntime tqdm

## Step 2: Download CodeSearchNet Dataset

In [ ]:
from datasets import load_dataset
import os

# Load CodeSearchNet (Python, JavaScript, Go, PHP, Ruby, Java)
print("Downloading CodeSearchNet dataset...")
print("This may take 10-20 minutes on first run.")

languages = ['python', 'javascript', 'go', 'php', 'ruby', 'java']
all_code = []

for lang in languages:
    print(f"Loading {lang}...")
    try:
        ds = load_dataset("code_search_net", lang, split="train", trust_remote_code=True)
        # Extract function code
        for item in ds:
            if item.get('func_code_string'):
                all_code.append({
                    'code': item['func_code_string'],
                    'language': lang
                })
        print(f"  {lang}: {len([x for x in all_code if x['language'] == lang])} samples")
    except Exception as e:
        print(f"  Error loading {lang}: {e}")

print(f"\nTotal samples: {len(all_code)}")

## Step 3: Build BPE Tokenizer

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, processors
import json

VOCAB_SIZE = 32000

print("Building BPE tokenizer...")

# Initialize BPE tokenizer
tokenizer = Tokenizer(models.BPE(unk_token="<UNK>"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

# Special tokens
special_tokens = ["<PAD>", "<UNK>", "<BOS>", "<EOS>", "<MASK>"]

# Train on code samples
trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=special_tokens,
    show_progress=True
)

# Create iterator for training
def code_iterator():
    for item in all_code:
        yield item['code']

tokenizer.train_from_iterator(code_iterator(), trainer=trainer)

# Add post-processor for special tokens
tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)

# Save tokenizer
tokenizer.save("arle_tokenizer.json")
print(f"Tokenizer saved with vocab size: {tokenizer.get_vocab_size()}")

# Test tokenizer
test_code = "def hello_world():\n    print('Hello!')"
encoded = tokenizer.encode(test_code)
print(f"\nTest encoding:")
print(f"  Input: {test_code}")
print(f"  Tokens: {encoded.tokens[:20]}...")
print(f"  IDs: {encoded.ids[:20]}...")

## Step 4: Prepare Training Data

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import random
from tqdm import tqdm

MAX_SEQ_LEN = 512  # Context window
BATCH_SIZE = 32

class CodeDataset(Dataset):
    def __init__(self, code_samples, tokenizer, max_len=MAX_SEQ_LEN):
        self.samples = []
        self.max_len = max_len
        
        print("Tokenizing samples...")
        for item in tqdm(code_samples):
            try:
                encoded = tokenizer.encode(item['code'])
                ids = encoded.ids
                
                # Create training samples (context -> next token)
                if len(ids) > 10:  # Skip very short samples
                    # Truncate or pad
                    if len(ids) > max_len:
                        ids = ids[:max_len]
                    self.samples.append(ids)
            except:
                continue
        
        print(f"Created {len(self.samples)} training samples")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        ids = self.samples[idx]
        
        # Input: all tokens except last
        # Target: all tokens except first (shifted by 1)
        input_ids = ids[:-1]
        target_ids = ids[1:]
        
        # Pad to max_len - 1
        pad_len = self.max_len - 1 - len(input_ids)
        if pad_len > 0:
            input_ids = input_ids + [0] * pad_len  # 0 is <PAD>
            target_ids = target_ids + [0] * pad_len
        
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'target_ids': torch.tensor(target_ids, dtype=torch.long)
        }

# Create dataset
dataset = CodeDataset(all_code, tokenizer)

# Split train/val
train_size = int(0.95 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## Step 5: Define LSTM Model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ArleLSTM(nn.Module):
    def __init__(self, vocab_size=32000, embed_dim=128, hidden_dim=256, num_layers=2, dropout=0.1):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # Positional encoding (learnable)
        self.pos_embedding = nn.Embedding(MAX_SEQ_LEN, embed_dim)
        
        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Attention layer
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim * 2,  # bidirectional
            num_heads=8,
            dropout=dropout,
            batch_first=True
        )
        
        # Output heads
        self.layer_norm = nn.LayerNorm(hidden_dim * 2)
        self.dropout = nn.Dropout(dropout)
        
        # Next token prediction head
        self.next_token_head = nn.Linear(hidden_dim * 2, vocab_size)
        
        # Rerank head (for scoring candidates)
        self.rerank_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        
        # Confidence head
        self.confidence_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for name, param in self.named_parameters():
            if 'weight' in name and param.dim() > 1:
                nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)
    
    def forward(self, input_ids, return_all_heads=False):
        batch_size, seq_len = input_ids.shape
        
        # Create position indices
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        
        # Embeddings
        x = self.embedding(input_ids) + self.pos_embedding(positions)
        x = self.dropout(x)
        
        # LSTM
        lstm_out, _ = self.lstm(x)
        
        # Self-attention
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        
        # Residual + LayerNorm
        x = self.layer_norm(lstm_out + attn_out)
        x = self.dropout(x)
        
        # Next token logits
        logits = self.next_token_head(x)
        
        if return_all_heads:
            # Use last position for rerank and confidence
            last_hidden = x[:, -1, :]
            rerank_score = self.rerank_head(last_hidden)
            confidence = self.confidence_head(last_hidden)
            return logits, rerank_score, confidence
        
        return logits

# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ArleLSTM(vocab_size=tokenizer.get_vocab_size()).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1e6:.1f} MB (FP32)")

## Step 6: Training Loop

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

# Training config
EPOCHS = 10
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding

# Training history
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    
    pbar = tqdm(loader, desc="Training")
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        target_ids = batch['target_ids'].to(device)
        
        optimizer.zero_grad()
        
        # Forward
        logits = model(input_ids)
        
        # Loss
        loss = criterion(logits.view(-1, logits.size(-1)), target_ids.view(-1))
        
        # Backward
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.6f}'})
    
    return total_loss / len(loader)

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in tqdm(loader, desc="Validating"):
        input_ids = batch['input_ids'].to(device)
        target_ids = batch['target_ids'].to(device)
        
        logits = model(input_ids)
        loss = criterion(logits.view(-1, logits.size(-1)), target_ids.view(-1))
        total_loss += loss.item()
        
        # Accuracy (non-padding tokens)
        preds = logits.argmax(dim=-1)
        mask = target_ids != 0
        correct += ((preds == target_ids) & mask).sum().item()
        total += mask.sum().item()
    
    acc = correct / total if total > 0 else 0
    return total_loss / len(loader), acc

# Training loop
print(f"\nStarting training on {device}")
print(f"Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}")
print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")
print("-" * 60)

best_val_loss = float('inf')
start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_acc': val_acc
        }, 'arle_best.pt')
        print("  Saved best model!")

elapsed = time.time() - start_time
print(f"\nTraining completed in {elapsed/3600:.1f} hours")
print(f"Best validation loss: {best_val_loss:.4f}")

## Step 7: Export to ONNX

In [ ]:
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

# Load best model
checkpoint = torch.load('arle_best.pt')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch'] + 1}")
print(f"Validation loss: {checkpoint['val_loss']:.4f}, accuracy: {checkpoint['val_acc']:.4f}")

# Create dummy input
dummy_input = torch.randint(0, 1000, (1, 256)).to(device)

# Export to ONNX
print("\nExporting to ONNX...")
model.cpu()
dummy_input = dummy_input.cpu()

torch.onnx.export(
    model,
    dummy_input,
    "arle.onnx",
    input_names=['input_ids'],
    output_names=['logits'],
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'seq_len'},
        'logits': {0: 'batch_size', 1: 'seq_len'}
    },
    opset_version=14
)

# Verify ONNX model
onnx_model = onnx.load("arle.onnx")
onnx.checker.check_model(onnx_model)
print("ONNX model verified!")

# Get file size
import os
onnx_size = os.path.getsize("arle.onnx") / 1e6
print(f"ONNX model size: {onnx_size:.1f} MB")

# Quantize to INT8
print("\nQuantizing to INT8...")
quantize_dynamic(
    "arle.onnx",
    "arle_int8.onnx",
    weight_type=QuantType.QUInt8
)

int8_size = os.path.getsize("arle_int8.onnx") / 1e6
print(f"INT8 model size: {int8_size:.1f} MB")
print(f"Compression: {onnx_size/int8_size:.1f}x")

## Step 8: Test the Model

In [ ]:
import onnxruntime as ort
import numpy as np

# Load INT8 model
session = ort.InferenceSession("arle_int8.onnx")

def predict_next_tokens(code_prefix, num_tokens=5):
    """Predict next tokens for given code prefix"""
    encoded = tokenizer.encode(code_prefix)
    input_ids = np.array([encoded.ids], dtype=np.int64)
    
    predictions = []
    for _ in range(num_tokens):
        # Run inference
        outputs = session.run(None, {'input_ids': input_ids})
        logits = outputs[0]
        
        # Get top prediction for last position
        next_token_logits = logits[0, -1, :]
        next_token_id = int(np.argmax(next_token_logits))
        
        # Decode token
        next_token = tokenizer.decode([next_token_id])
        predictions.append(next_token)
        
        # Append to input for next iteration
        input_ids = np.concatenate([input_ids, [[next_token_id]]], axis=1)
    
    return predictions

# Test predictions
test_cases = [
    "def calculate_",
    "class User",
    "import ",
    "for item in ",
    "if __name__ == "
]

print("Testing model predictions:\n")
for test in test_cases:
    predictions = predict_next_tokens(test, num_tokens=10)
    completion = ''.join(predictions)
    print(f"Input: '{test}'")
    print(f"Completion: '{completion}'")
    print()

## Step 9: Download Files

Run this cell to download the trained model and tokenizer:

In [ ]:
from google.colab import files

print("Downloading files...")
print("\n1. Downloading arle_int8.onnx (model)")
files.download('arle_int8.onnx')

print("\n2. Downloading arle_tokenizer.json (tokenizer)")
files.download('arle_tokenizer.json')

print("\n" + "="*60)
print("DONE! Place these files in:")
print("  ~/.arlecchino/models/arle_int8.onnx")
print("  ~/.arlecchino/models/arle_tokenizer.json")
print("="*60)

## Troubleshooting

**Session disconnected?**
- Colab disconnects after ~90 min of inactivity
- Keep the tab open and active
- Or run this in a cell to keep alive:
```javascript
function ClickConnect(){
  console.log("Keeping alive...");
  document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)
```

**Out of memory?**
- Reduce BATCH_SIZE to 16
- Reduce MAX_SEQ_LEN to 256

**Training too slow?**
- Make sure GPU is enabled (Runtime -> Change runtime type -> T4)
- Reduce number of epochs